Ten notatnik tworzy aplikację Streamlit wykorzystującą framework CrewAI, która automatycznie przygotowuje materiały przed spotkaniem biznesowym poprzez współpracę czterech agentów AI analizujących kontekst, branżę, strategię i tworzących brief dla kierownictwa. Plik demonstruje jak uruchomić  aplikację lokalnie i udostępnić ją przez localtunnel.

# Setup

In [ ]:
!pip install -qq  crewai crewai_tools streamlit

In [ ]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

`localtunnel` to narzędzie wiersza poleceń, które tworzy publicznie dostępny URL do lokalnego serwera działającego na komputerze użytkownika. Jest przydatne podczas testowania aplikacji webowych lub udostępniania ich innym osobom bez konieczności wdrażania na zdalnym serwerze.

# App

In [ ]:
%%writefile muh_app.py

import streamlit as st
from crewai import Agent, Task, Crew, LLM
from crewai.process import Process
from crewai_tools import SerperDevTool
import os


# inicjalizacja aplikacji
st.set_page_config(page_title="AI Meeting Agent 📝", layout="wide")
st.title("Agentura przed spotkaniem 📝")


# API
st.sidebar.header("API")
openai_key = st.sidebar.text_input("OpenAI API Key", type="password")
serper_key = st.sidebar.text_input("Serper API Key", type="password")

os.environ["OPENAI_API_KEY"] = openai_key
os.environ["SERPER_API_KEY"] = serper_key

# konfiguracja narzedzi
muh_llm = LLM(model= "gpt-4o-mini" , temperature= 0.7, api_key=openai_key)
search_tool = SerperDevTool()

# parametry
company_name = st.text_input("Wprowadź nazwę firmy:")
meeting_objective = st.text_input("Wprowadź cel spotkania:")
attendees = st.text_area("Wprowadź uczestników i ich role (po jednym w linii):")
meeting_duration = st.number_input("Wprowadź czas trwania spotkania (w minutach):", min_value=15, max_value=180, value=60, step=15)
focus_areas = st.text_input("Wprowadź konkretne obszary zainteresowania lub obawy:")

# agenci
context_analyzer = Agent(
    role='Specjalista ds. Kontekstu ',
    goal='Analizuj i podsumowuj kluczowe informacje kontekstowe do spotkania',
    backstory='Jesteś ekspertem w szybkim rozumieniu złożonych kontekstów biznesowych i identyfikowaniu kluczowych informacji.',
    verbose=True,
    allow_delegation=False,
    llm = muh_llm,
    tools=[search_tool]
)

industry_insights_generator = Agent(
    role='Ekspert Branżowy',
    goal='Dostarcz dogłębną analizę branży i zidentyfikuj kluczowe trendy',
    backstory='Jesteś doświadczonym analitykiem branżowym ze zdolnością do dostrzegania pojawiających się trendów i możliwości.',
    verbose=True,
    allow_delegation=False,
    llm = muh_llm,
    tools=[search_tool]
)

strategy_formulator = Agent(
    role='Strateg Spotkania',
    goal='Opracuj dostosowaną strategię spotkania i szczegółową agendę',
    backstory='Jesteś mistrzem planowania spotkań, znanym z tworzenia wysoce efektywnych strategii i agend.',
    verbose=True,
    allow_delegation=False,
    llm = muh_llm,
)

executive_briefing_creator = Agent(
    role='Specjalista ds. Komunikacji',
    goal='Zsyntetyzuj informacje w zwięzłe i skuteczne briefingi',
    backstory='Jesteś ekspertem komunikacji, biegłym w przekładaniu złożonych informacji na jasne, praktyczne wskazówki.',
    verbose=True,
    allow_delegation=False,
    llm = muh_llm,
)

# zadania
context_analysis_task = Task(
    description=f"""
    Przeanalizuj kontekst spotkania z firmą {company_name}, biorąc pod uwagę:
    1. Cel spotkania: {meeting_objective}
    2. Uczestników: {attendees}
    3. Czas trwania spotkania: {meeting_duration} minut
    4. Konkretne obszary zainteresowania lub obawy: {focus_areas}

    Dokładnie zbadaj firmę {company_name}, w tym:
    1. Ostatnie wiadomości i komunikaty prasowe
    2. Kluczowe produkty lub usługi
    3. Głównych konkurentów

    Przedstaw kompleksowe podsumowanie swoich ustaleń, podkreślając najistotniejsze informacje dla kontekstu spotkania.
    Sformatuj wynik używając markdown z odpowiednimi nagłówkami i podtytułami.
    """,
    agent=context_analyzer,
    expected_output="Szczegółowa analiza kontekstu spotkania i tła firmy, zawierająca ostatnie wydarzenia, wyniki finansowe i znaczenie dla celu spotkania, sformatowana w markdown z nagłówkami i podtytułami."
)

industry_analysis_task = Task(
      description=f"""
      Na podstawie analizy kontekstu dla firmy {company_name} i celu spotkania: {meeting_objective}, przedstaw dogłębną analizę branży:
      1. Zidentyfikuj kluczowe trendy i wydarzenia w branży
      2. Przeanalizuj sytuację konkurencyjną
      3. Podkreśl potencjalne szanse i zagrożenia
      4. Przedstaw spostrzeżenia dotyczące pozycjonowania rynkowego

      Upewnij się, że analiza jest odpowiednia do celu spotkania i ról uczestników.
      Sformatuj wynik używając markdown z odpowiednimi nagłówkami i podtytułami.
      """,
      agent=industry_insights_generator,
      expected_output="Kompleksowy raport analizy branży, zawierający trendy, sytuację konkurencyjną, szanse, zagrożenia i istotne spostrzeżenia dla celu spotkania, sformatowany w markdown z nagłówkami i podtytułami."
  )

strategy_development_task = Task(
      description=f"""
      Wykorzystując analizę kontekstu i spostrzeżenia branżowe, opracuj dostosowaną strategię spotkania i szczegółową agendę dla {meeting_duration}-minutowego spotkania z firmą {company_name}. Zawrzyj:
      1. Agendę z określonymi ramami czasowymi i jasnymi celami dla każdej sekcji
      2. Kluczowe punkty do omówienia dla każdego elementu agendy
      3. Proponowanych mówców lub prowadzących dla każdej sekcji
      4. Potencjalne tematy do dyskusji i pytania napędzające rozmowę
      5. Strategie dotyczące konkretnych obszarów zainteresowania i obaw: {focus_areas}

      Upewnij się, że strategia i agenda są zgodne z celem spotkania: {meeting_objective}
      Sformatuj wynik używając markdown z odpowiednimi nagłówkami i podtytułami.
      """,
      agent=strategy_formulator,
      expected_output="Szczegółowa strategia spotkania i agenda z ramami czasowymi, zawierająca cele, kluczowe punkty do omówienia i strategie dotyczące konkretnych obszarów zainteresowania, sformatowana w markdown z nagłówkami i podtytułami."
  )

executive_brief_task = Task(
      description=f"""
      Dokonaj syntezy wszystkich zebranych informacji w kompleksowy, lecz zwięzły brief dla kierownictwa na spotkanie z firmą {company_name}. Utwórz następujące komponenty:

      1. Szczegółowe jednostronicowe podsumowanie dla kierownictwa zawierające:
          - Jasne określenie celu spotkania
          - Lista kluczowych uczestników i ich ról
          - Najważniejsze informacje o tle firmy {company_name} i istotny kontekst branżowy
          - Najważniejsze 3-5 celów strategicznych spotkania, zgodnych z głównym celem
          - Krótki przegląd struktury spotkania i kluczowych tematów do omówienia

      2. Szczegółowa lista kluczowych punktów do omówienia, każdy poparty:
          - Istotnymi danymi lub statystykami
          - Konkretnymi przykładami lub studiami przypadków
          - Powiązaniem z obecną sytuacją lub wyzwaniami firmy

      3. Przewidywanie i przygotowanie się na potencjalne pytania:
          - Lista prawdopodobnych pytań od uczestników na podstawie ich ról i celu spotkania
          - Przygotowanie przemyślanych, opartych na danych odpowiedzi na każde pytanie
          - Dołączenie wszelkich dodatkowych informacji lub kontekstu, które mogą być potrzebne

      4. Rekomendacje strategiczne i następne kroki:
          - Przedstawienie 3-5 możliwych do wdrożenia rekomendacji na podstawie analizy
          - Nakreślenie jasnych następnych kroków do wdrożenia lub działań następczych
          - Zaproponowanie harmonogramów lub terminów dla kluczowych działań
          - Identyfikacja potencjalnych wyzwań lub przeszkód i propozycje strategii ich łagodzenia

      Upewnij się, że brief jest kompleksowy, ale zwięzły, możliwy do wdrożenia i dokładnie zgodny z celem spotkania: {meeting_objective}. Dokument powinien być ustrukturyzowany dla łatwej nawigacji i szybkiego odniesienia podczas spotkania.
      Sformatuj wynik używając markdown z odpowiednimi nagłówkami i podtytułami.
      """,
      agent=executive_briefing_creator,
      expected_output="Kompleksowy brief dla kierownictwa zawierający podsumowanie, kluczowe punkty do omówienia, przygotowanie pytań i odpowiedzi oraz \
      rekomendacje strategiczne, sformatowany w markdown z głównymi nagłówkami (H1), nagłówkami sekcji (H2) i nagłówkami podsekcji (H3) tam, gdzie to właściwe.\
       Użyj punktów, list numerowanych i wyróżnień (pogrubienie/kursywa) dla kluczowych informacji."
  )

# sklad
meeting_prep_crew = Crew(
      agents=[context_analyzer, industry_insights_generator, strategy_formulator, executive_briefing_creator],
      tasks=[context_analysis_task, industry_analysis_task, strategy_development_task, executive_brief_task],
      verbose=True,
      process=Process.sequential
  )

# odpal po nacisnieciu guzika ;-)
if st.button("Przygotuj Spotkanie"):
    with st.spinner("Agenci AI przygotowują twoje spotkanie..."):
        result = meeting_prep_crew.kickoff()
    st.markdown(result)

st.sidebar.markdown("""
## Jak korzystać z aplikacji:
1. Wprowadź swoje klucze API w pasku bocznym.
2. Podaj wymagane informacje o spotkaniu.
3. Kliknij 'Przygotuj Spotkanie', aby wygenerować kompleksowy pakiet przygotowawczy do spotkania.

Agenci AI będą współpracować, aby:
- Przeanalizować kontekst spotkania i tło firmy
- Dostarczyć spostrzeżenia i trendy branżowe
- Opracować dostosowaną strategię spotkania i agendę
- Stworzyć brief dla kierownictwa z kluczowymi punktami do omówienia

Ten proces może potrwać kilka minut. Prosimy o cierpliwość!
""")


Overwriting muh_app.py


Ten kod definiuje aplikację webową stworzoną za pomocą biblioteki Streamlit, która wykorzystuje framework CrewAI do automatycznego przygotowania materiałów przed spotkaniem biznesowym. Aplikacja pobiera od użytkownika informacje dotyczące spotkania (nazwę firmy, cel, uczestników, czas trwania, obszary zainteresowań) i następnie uruchamia sekwencję agentów AI w celu wygenerowania analizy kontekstu, spostrzeżeń branżowych, strategii spotkania oraz briefu dla kierownictwa.

**Szczegółowy opis:**

1.  **Import bibliotek:** Kod zaczyna od importu niezbędnych bibliotek:
    *   `streamlit`: Do tworzenia interfejsu użytkownika aplikacji webowej.
    *   `crewai`: Główna biblioteka do pracy z agentami AI i zadaniami.
    *   `crewai_tools`: Zawiera narzędzia, które agenci mogą wykorzystywać (np. wyszukiwarka).
    *   `os`: Do interakcji z systemem operacyjnym, w tym ustawiania zmiennych środowiskowych.

2.  **Konfiguracja aplikacji Streamlit:** Ustawia tytuł strony i układ (`wide`).

3.  **Pobieranie kluczy API:** Aplikacja posiada pasek boczny, w którym użytkownik może wprowadzić klucze API dla OpenAI (model językowy) i Serper (narzędzie do wyszukiwania). Klucze te są następnie zapisywane jako zmienne środowiskowe (`os.environ`).

4.  **Konfiguracja LLM i narzędzi:** Inicjalizuje model językowy `gpt-4o-mini` z określoną temperaturą (kontrola losowości generowanych odpowiedzi) oraz kluczem API. Tworzy instancję narzędzia `SerperDevTool`, które umożliwia agentom wyszukiwanie informacji w internecie.

5.  **Pobieranie danych wejściowych od użytkownika:** Aplikacja wyświetla pola tekstowe i numeryczne, w których użytkownik wprowadza informacje o spotkaniu: nazwę firmy, cel spotkania, listę uczestników z ich rolami, czas trwania spotkania oraz obszary zainteresowań.

6.  **Definicja agentów AI:** Definiuje cztery agenty AI, każdy z określoną rolą, celem i historią (backstory):
    *   `context_analyzer`: Analizuje kontekst spotkania i firmę.
    *   `industry_insights_generator`: Dostarcza analizy branżowe.
    *   `strategy_formulator`: Opracowuje strategię spotkania i agendę.
    *   `executive_briefing_creator`: Tworzy brief dla kierownictwa.

    Każdy agent ma przypisany model językowy (`muh_llm`) oraz dostęp do narzędzia `search_tool`.  Parametr `allow_delegation=False` oznacza, że agenci nie mogą przekazywać zadań innym agentom.

7.  **Definicja zadań:** Definiuje cztery zadania, które mają zostać wykonane przez agentów:
    *   `context_analysis_task`: Analiza kontekstu spotkania i firmy.
    *   `industry_analysis_task`: Analiza branży.
    *   `strategy_development_task`: Opracowanie strategii spotkania i agendy.
    *   `executive_brief_task`: Stworzenie briefu dla kierownictwa.

    Każde zadanie ma opis, przypisanego agenta oraz oczekiwany format wyjściowy (markdown). Opisy zadań zawierają dynamicznie wstawiane dane wejściowe od użytkownika.

8.  **Tworzenie Crew:** Tworzy `Crew` składający się z czterech agentów i czterech zadań. Ustawia tryb przetwarzania na sekwencyjny (`Process.sequential`), co oznacza, że zadania będą wykonywane po kolei.

9.  **Uruchomienie procesu przygotowania spotkania:** Po naciśnięciu przycisku "Przygotuj Spotkanie", aplikacja uruchamia proces przygotowania spotkania za pomocą metody `kickoff()`. Wyświetla spinner podczas trwania procesu. Wynik działania Crew (wygenerowany raport) jest wyświetlany w formacie markdown.

10. **Instrukcja obsługi:** W pasku bocznym znajduje się instrukcja obsługi aplikacji, która wyjaśnia, jak korzystać z interfejsu i jakie kroki podejmują agenci AI.



W skrócie, aplikacja ta automatyzuje proces przygotowania do spotkań biznesowych, wykorzystując agentów AI do analizy kontekstu, dostarczania spostrzeżeń branżowych oraz tworzenia strategii i materiałów informacyjnych.

# Biegnij Forrest

In [ ]:
!streamlit run muh_app.py --server.address=localhost  &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.75.208.4
⠙your url is: https://dry-doodles-mate.loca.lt
